# Platform Observability: Step 2 Infrastructure Setup

Creates everything dbt Cloud needs to build and maintain the `PLATFORM_OBSERVABILITY` history database.

**What this creates**

| Object | Name | Purpose |
|---|---|---|
| Role | `OBSERVABILITY_TRANSFORMER` | Used by dbt Cloud; owns the schemas and builds all models |
| Role | `OBSERVABILITY_READER` | Read-only access to `HISTORY` and `MARTS` for analysts and BI tools |
| Warehouse | `OBSERVABILITY_WH` | XS, 60-second auto-suspend |
| Resource monitor | `OBSERVABILITY_RM` | Monthly credit cap on the warehouse |
| Database | `PLATFORM_OBSERVABILITY` | Hosts the history data |
| Schemas | `STAGING`, `HISTORY`, `MARTS` | Views over source, permanent history, reporting summaries |
| Service user | `DBT_CLOUD_SVC` | dbt Cloud deployment login, key-pair authentication |

**Role hierarchy**

```
ACCOUNTADMIN
└── SYSADMIN
    └── OBSERVABILITY_TRANSFORMER   (dbt Cloud, developers)
        └── OBSERVABILITY_READER    (analysts, BI)
```

**Before you run**
1. Generate a key pair for the service user (see the next cell) and paste the public key into section 6.
2. Replace `<YOUR_USER_NAME>` in section 7 with your Snowflake login name.
3. Run as a user who holds `ACCOUNTADMIN`.

**Where to run it:** Snowflake Notebooks run under a fixed owner role, and this setup deliberately switches between `USERADMIN`, `SYSADMIN`, `SECURITYADMIN` and `ACCOUNTADMIN` to follow least privilege. Run the companion script `setup/02_infrastructure_setup.sql` in a Snowsight SQL worksheet or SnowSQL. This notebook documents the same statements cell by cell.

Every statement uses `IF NOT EXISTS` or is a grant, so the script is safe to re-run.

## 0. Generate the dbt Cloud key pair (run locally, not in Snowflake)

Snowflake service users (`TYPE = SERVICE`) cannot log in with a password, so dbt Cloud authenticates with a key pair.

```bash
# Private key (encrypted; you will be prompted for a passphrase)
openssl genrsa 2048 | openssl pkcs8 -topk8 -v2 des3 -inform PEM -out dbt_cloud_svc_key.p8

# Public key
openssl rsa -in dbt_cloud_svc_key.p8 -pubout -out dbt_cloud_svc_key.pub
```

- Paste the **public** key body (without the `-----BEGIN/END PUBLIC KEY-----` lines) into section 6.
- Upload the **private** key and passphrase to dbt Cloud in Step 3.
- Never commit either key file to this repository.

## 1. Roles

`USERADMIN` creates roles. Both roles roll up to `SYSADMIN`, so platform administrators keep visibility into everything this project creates.

In [ ]:
USE ROLE USERADMIN;

CREATE ROLE IF NOT EXISTS OBSERVABILITY_TRANSFORMER
  COMMENT = 'dbt Cloud role: builds and owns PLATFORM_OBSERVABILITY';

CREATE ROLE IF NOT EXISTS OBSERVABILITY_READER
  COMMENT = 'Read-only access to PLATFORM_OBSERVABILITY history and marts';

GRANT ROLE OBSERVABILITY_READER      TO ROLE OBSERVABILITY_TRANSFORMER;
GRANT ROLE OBSERVABILITY_TRANSFORMER TO ROLE SYSADMIN;

## 2. Warehouse and resource monitor

An XS warehouse is plenty for daily incremental loads. The one-hour statement timeout stops a runaway query, and the resource monitor caps monthly spend.

**Adjust before the initial backfill:** loading roughly a year of query history with full SQL text can take longer than an hour on XS. For that one run, temporarily raise the size (`ALTER WAREHOUSE OBSERVABILITY_WH SET WAREHOUSE_SIZE = 'SMALL'`) or the timeout, then set it back.

In [ ]:
USE ROLE SYSADMIN;

CREATE WAREHOUSE IF NOT EXISTS OBSERVABILITY_WH
  WITH WAREHOUSE_SIZE              = 'XSMALL'
       AUTO_SUSPEND                = 60
       AUTO_RESUME                 = TRUE
       INITIALLY_SUSPENDED         = TRUE
       STATEMENT_TIMEOUT_IN_SECONDS = 3600
       COMMENT                     = 'Compute for PLATFORM_OBSERVABILITY dbt runs';

USE ROLE ACCOUNTADMIN;

CREATE RESOURCE MONITOR IF NOT EXISTS OBSERVABILITY_RM
  WITH CREDIT_QUOTA    = 20
       FREQUENCY       = MONTHLY
       START_TIMESTAMP = IMMEDIATELY
       TRIGGERS ON  80 PERCENT DO NOTIFY
                ON 100 PERCENT DO SUSPEND
                ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE OBSERVABILITY_WH SET RESOURCE_MONITOR = OBSERVABILITY_RM;

## 3. Database and schemas

- **`STAGING`** holds views over `ACCOUNT_USAGE`; nothing is stored there.
- **`HISTORY`** is the irreplaceable layer: once Snowflake ages a row out of `ACCOUNT_USAGE`, this is the only copy. It gets **30 days of Time Travel** (Enterprise allows up to 90) as a safety net against an accidental drop or bad load.
- **`MARTS`** holds rebuildable summaries, so it keeps the 7-day database default.

In [ ]:
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS PLATFORM_OBSERVABILITY
  DATA_RETENTION_TIME_IN_DAYS = 7
  COMMENT = 'Long-term history of Snowflake object changes, query activity and Cortex usage';

CREATE SCHEMA IF NOT EXISTS PLATFORM_OBSERVABILITY.STAGING
  COMMENT = 'dbt staging views over SNOWFLAKE.ACCOUNT_USAGE';

CREATE SCHEMA IF NOT EXISTS PLATFORM_OBSERVABILITY.HISTORY
  DATA_RETENTION_TIME_IN_DAYS = 30
  COMMENT = 'Permanent append-only history tables (2-year retention)';

CREATE SCHEMA IF NOT EXISTS PLATFORM_OBSERVABILITY.MARTS
  COMMENT = 'Daily summaries for reporting';

## 4. Hand the schemas to dbt

`OBSERVABILITY_TRANSFORMER` takes ownership of the three schemas so dbt can create, replace and alter models without extra grants. It also gets `CREATE SCHEMA` on the database, because dbt Cloud development environments build into a personal schema (for example `DBT_AKIRUBAKARAN`) so your experiments never touch production tables.

The database itself stays owned by `SYSADMIN`.

In [ ]:
USE ROLE SYSADMIN;

GRANT USAGE, MONITOR       ON DATABASE PLATFORM_OBSERVABILITY TO ROLE OBSERVABILITY_TRANSFORMER;
GRANT CREATE SCHEMA        ON DATABASE PLATFORM_OBSERVABILITY TO ROLE OBSERVABILITY_TRANSFORMER;

GRANT OWNERSHIP ON SCHEMA PLATFORM_OBSERVABILITY.STAGING TO ROLE OBSERVABILITY_TRANSFORMER COPY CURRENT GRANTS;
GRANT OWNERSHIP ON SCHEMA PLATFORM_OBSERVABILITY.HISTORY TO ROLE OBSERVABILITY_TRANSFORMER COPY CURRENT GRANTS;
GRANT OWNERSHIP ON SCHEMA PLATFORM_OBSERVABILITY.MARTS   TO ROLE OBSERVABILITY_TRANSFORMER COPY CURRENT GRANTS;

GRANT USAGE, OPERATE ON WAREHOUSE OBSERVABILITY_WH TO ROLE OBSERVABILITY_TRANSFORMER;
GRANT USAGE          ON WAREHOUSE OBSERVABILITY_WH TO ROLE OBSERVABILITY_READER;

## 5. Reader access (including tables dbt has not built yet)

Future grants mean every table or view dbt later creates in `HISTORY` and `MARTS` is automatically readable by `OBSERVABILITY_READER`. `STAGING` is intentionally excluded: analysts should query history, not raw source views.

In [ ]:
USE ROLE SECURITYADMIN;

GRANT USAGE ON DATABASE PLATFORM_OBSERVABILITY         TO ROLE OBSERVABILITY_READER;
GRANT USAGE ON SCHEMA   PLATFORM_OBSERVABILITY.HISTORY TO ROLE OBSERVABILITY_READER;
GRANT USAGE ON SCHEMA   PLATFORM_OBSERVABILITY.MARTS   TO ROLE OBSERVABILITY_READER;

GRANT SELECT ON FUTURE TABLES IN SCHEMA PLATFORM_OBSERVABILITY.HISTORY TO ROLE OBSERVABILITY_READER;
GRANT SELECT ON FUTURE VIEWS  IN SCHEMA PLATFORM_OBSERVABILITY.HISTORY TO ROLE OBSERVABILITY_READER;
GRANT SELECT ON FUTURE TABLES IN SCHEMA PLATFORM_OBSERVABILITY.MARTS   TO ROLE OBSERVABILITY_READER;
GRANT SELECT ON FUTURE VIEWS  IN SCHEMA PLATFORM_OBSERVABILITY.MARTS   TO ROLE OBSERVABILITY_READER;

## 6. Access to `SNOWFLAKE.ACCOUNT_USAGE`

dbt reads every source from the shared `SNOWFLAKE` database. `IMPORTED PRIVILEGES` is the simplest grant that covers all the views we need, including the Cortex usage views.

**Least-privilege alternative:** grant the narrower Snowflake database roles instead (`SNOWFLAKE.GOVERNANCE_VIEWER` for query and access history, `SNOWFLAKE.USAGE_VIEWER` for metering, `SNOWFLAKE.OBJECT_VIEWER` for object metadata). Check that the Cortex views from Step 1 are covered before switching.

In [ ]:
USE ROLE ACCOUNTADMIN;

GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE OBSERVABILITY_TRANSFORMER;

## 7. dbt Cloud service user and developer access

- **`DBT_CLOUD_SVC`** is a `TYPE = SERVICE` user: no password, no MFA prompts, key-pair login only. dbt Cloud's scheduled production jobs run as this user.
- **Your own user** gets `OBSERVABILITY_TRANSFORMER` so you can develop in the dbt Cloud IDE with your own credentials.

Replace both placeholders before running.

*Optional hardening:* attach a network policy that allows only dbt Cloud's published IP addresses for your dbt Cloud region.

In [ ]:
USE ROLE USERADMIN;

CREATE USER IF NOT EXISTS DBT_CLOUD_SVC
  TYPE              = SERVICE
  DEFAULT_ROLE      = OBSERVABILITY_TRANSFORMER
  DEFAULT_WAREHOUSE = OBSERVABILITY_WH
  DEFAULT_NAMESPACE = 'PLATFORM_OBSERVABILITY.STAGING'
  RSA_PUBLIC_KEY    = '<DBT_CLOUD_PUBLIC_KEY>'
  COMMENT           = 'dbt Cloud production jobs for PLATFORM_OBSERVABILITY';

USE ROLE SECURITYADMIN;

GRANT ROLE OBSERVABILITY_TRANSFORMER TO USER DBT_CLOUD_SVC;
GRANT ROLE OBSERVABILITY_TRANSFORMER TO USER <YOUR_USER_NAME>;

## 8. Validate: grants and user setup

**Expected:**
- `OBSERVABILITY_TRANSFORMER` shows `OWNERSHIP` on the three schemas, `USAGE`/`OPERATE` on the warehouse, `USAGE`/`MONITOR`/`CREATE SCHEMA` on the database, and `IMPORTED PRIVILEGES` on `SNOWFLAKE`.
- `DBT_CLOUD_SVC` shows `TYPE = SERVICE` and a populated `RSA_PUBLIC_KEY_FP` (fingerprint). An empty fingerprint means the public key was not set.

In [ ]:
USE ROLE SECURITYADMIN;

SHOW GRANTS TO ROLE OBSERVABILITY_TRANSFORMER;
SHOW GRANTS TO ROLE OBSERVABILITY_READER;
SHOW FUTURE GRANTS IN SCHEMA PLATFORM_OBSERVABILITY.HISTORY;

DESC USER DBT_CLOUD_SVC;

## 9. Validate: act as dbt would

Switch to the dbt role and prove it can (a) read `ACCOUNT_USAGE`, and (b) create and drop a table in each schema it will build into.

**Expected:** a non-zero query count, and every `CREATE`/`DROP` succeeds. Any `insufficient privileges` error points back to the section that failed.

In [ ]:
USE ROLE OBSERVABILITY_TRANSFORMER;
USE WAREHOUSE OBSERVABILITY_WH;

SELECT COUNT(*) AS queries_last_24h
FROM   SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE  start_time >= DATEADD(hour, -24, CURRENT_TIMESTAMP());

CREATE TABLE PLATFORM_OBSERVABILITY.STAGING.ZZ_SETUP_CHECK (id INT);
CREATE TABLE PLATFORM_OBSERVABILITY.HISTORY.ZZ_SETUP_CHECK (id INT);
CREATE TABLE PLATFORM_OBSERVABILITY.MARTS.ZZ_SETUP_CHECK   (id INT);

DROP TABLE PLATFORM_OBSERVABILITY.STAGING.ZZ_SETUP_CHECK;
DROP TABLE PLATFORM_OBSERVABILITY.HISTORY.ZZ_SETUP_CHECK;
DROP TABLE PLATFORM_OBSERVABILITY.MARTS.ZZ_SETUP_CHECK;

ALTER WAREHOUSE OBSERVABILITY_WH SUSPEND;

## Rollback (commented out)

Removes everything this step created. Only uncomment if you need to start over; once Step 7 loads history, dropping the database destroys data Snowflake may no longer hold.

In [ ]:
-- USE ROLE ACCOUNTADMIN;
-- DROP USER             IF EXISTS DBT_CLOUD_SVC;
-- DROP DATABASE         IF EXISTS PLATFORM_OBSERVABILITY;
-- DROP WAREHOUSE        IF EXISTS OBSERVABILITY_WH;
-- DROP RESOURCE MONITOR IF EXISTS OBSERVABILITY_RM;
-- DROP ROLE             IF EXISTS OBSERVABILITY_READER;
-- DROP ROLE             IF EXISTS OBSERVABILITY_TRANSFORMER;

## Next step: dbt Cloud connection (Step 3)

With validation passing, the dbt Cloud project connection uses:

| Setting | Value |
|---|---|
| Account | Your account identifier, `<orgname>-<account_name>` (Step 1 of `snowflake_config_basics.sql` shows both) |
| Database | `PLATFORM_OBSERVABILITY` |
| Warehouse | `OBSERVABILITY_WH` |
| Role | `OBSERVABILITY_TRANSFORMER` |
| Deployment user | `DBT_CLOUD_SVC`, auth method **Key pair**, private key from section 0 |
| Development user | Your own login, target schema e.g. `DBT_<YOURNAME>` |

**A safeguard Step 3 will add:** every `HISTORY` model will be configured with `full_refresh: false`. Without it, a `dbt run --full-refresh` would rebuild history tables from `ACCOUNT_USAGE`, silently discarding anything older than Snowflake's 365-day window.